# ROC curves
Generates ROC curve figures for SKF and LOSO experiments.

SKF produces one figure per feature strategy, with all four models overlaid. LOSO produces one figure per feature strategy per model, with individual site curves and a mean curve.

Execute cells top to bottom. Only cells marked **[CONFIGURE]** require changes.

> Run `run_experiments.ipynb` before running this notebook.

## 1. Environment Setup **[OPTIONAL]**
Mounts Google Drive and installs dependencies when running on Colab. Skip if running locally.

> **NOTE: requires moving `PROJECT` folder to Google Drive.**

In [ ]:
# COLAB
import sys
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd '/content/drive/My Drive/PROJECT'

    !pip install -q numpy matplotlib

    sys.path.insert(0, './product/src')

## 2. Imports

In [ ]:
import os
import sys
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker
from matplotlib.lines import Line2D
from pathlib import Path
from utils.paths import get_project_root
from visualisation.common import discover_timestamps, auc_from_roc
from visualisation.roc_curves import interpolate_roc, site_label, load_raw, plot_skf_roc, plot_loso_roc

## 3. Configuration **[CONFIGURE]**

The `RUNS` table maps each `(model, dim)` combination to a `(folder_name, tags)` pair. Each tag assigns a `'CV-Experiment'` label to a timestamp index discovered in that folder. Valid tag values are: `'SKF-Baseline'`, `'SKF-Optimised'`, `'LOSO-Baseline'`, `'LOSO-Optimised'`.

Use `FILTER` to restrict which figures are generated. Set any entry to `None` to include everything.

| Variable | Description |
|---|---|
| `folder_name` | The `model_name` string passed to `CentralisedLogger` |
| `tags` | Maps timestamp index (0 = oldest) to a `'CV-Experiment'` label |

In [ ]:
# ---- CHANGE THESE ------------------------------------------------
# (model, dim) -> (folder_name, tags)
# Run Section 4 first to see which index maps to which timestamp.
RUNS = {
    ('SVM', 'SelectKBest'): ('svm_selectkbest', {0: 'SKF-Baseline', 1: 'LOSO-Baseline', 2: 'SKF-Optimised', 3: 'LOSO-Optimised'}),
    ('SVM', 'PCA'):         ('svm_pca',          {0: 'SKF-Baseline', 1: 'LOSO-Baseline', 2: 'SKF-Optimised', 3: 'LOSO-Optimised'}),
    ('SVM', 'SDAE'):        ('svm_sdae',          {0: 'SKF-Baseline', 1: 'LOSO-Baseline', 2: 'SKF-Optimised', 3: 'LOSO-Optimised'}),
    ('RF',  'SelectKBest'): ('rf_selectkbest',   {0: 'SKF-Baseline', 1: 'LOSO-Baseline', 2: 'SKF-Optimised', 3: 'LOSO-Optimised'}),
    ('RF',  'PCA'):         ('rf_pca',            {0: 'SKF-Baseline', 1: 'LOSO-Baseline', 2: 'SKF-Optimised', 3: 'LOSO-Optimised'}),
    ('RF',  'SDAE'):        ('rf_sdae',            {0: 'SKF-Baseline', 1: 'LOSO-Baseline', 2: 'SKF-Optimised', 3: 'LOSO-Optimised'}),
    ('XGB', 'SelectKBest'): ('xgb_selectkbest',  {0: 'SKF-Baseline', 1: 'LOSO-Baseline', 2: 'SKF-Optimised', 3: 'LOSO-Optimised'}),
    ('XGB', 'PCA'):         ('xgb_pca',           {0: 'SKF-Baseline', 1: 'LOSO-Baseline', 2: 'SKF-Optimised', 3: 'LOSO-Optimised'}),
    ('XGB', 'SDAE'):        ('xgb_sdae',           {0: 'SKF-Baseline', 1: 'LOSO-Baseline', 2: 'SKF-Optimised', 3: 'LOSO-Optimised'}),
    ('GCN', 'SelectKBest'): ('gcn_selectkbest',  {0: 'SKF-Baseline', 1: 'LOSO-Baseline', 2: 'SKF-Optimised', 3: 'LOSO-Optimised'}),
    ('GCN', 'PCA'):         ('gcn_pca',           {0: 'SKF-Baseline', 1: 'LOSO-Baseline', 2: 'SKF-Optimised', 3: 'LOSO-Optimised'}),
    ('GCN', 'SDAE'):        ('gcn_sdae',           {0: 'SKF-Baseline', 1: 'LOSO-Baseline', 2: 'SKF-Optimised', 3: 'LOSO-Optimised'}),
}

MODEL_COLOURS = {
    'SVM': '#4477AA',
    'RF':  '#EE6677',
    'XGB': '#228833',
    'GCN': '#CCBB44',
}

DIM_LABELS = {
    'SelectKBest': 'SelectKBest',
    'PCA':         'PCA',
    'SDAE':        'SDAE',
}

# Filter which figures to generate.
# Set any entry to None to include all options for that dimension.
FILTER = {
    'experiments': None,   # None or e.g. ['Optimised']
    'cv':          None,   # None or e.g. ['SKF']
    'models':      None,   # None or e.g. ['SVM', 'GCN']
    'dims':        None,   # None or e.g. ['SelectKBest']
}

## 4. Discover Timestamps **[CONFIGURE]**

Scans each logger folder and prints all discovered timestamps with their index.

In [ ]:
seen_folders = set()
for (model, dim), (folder_name, tags) in RUNS.items():
    if folder_name in seen_folders:
        continue
    seen_folders.add(folder_name)
    timestamps = discover_timestamps(folder_name)
    print(f"{folder_name}")
    if not timestamps:
        print('  (no runs found)')
    for i, ts in enumerate(timestamps):
        current_tag = tags.get(i, '(untagged)')
        print(f'  [{i}]  {ts}  ->  {current_tag}')
    print()

## 5. Helper Functions

`auc_from_roc` computes AUC via the trapezoidal rule. `interpolate_roc` resamples a curve onto a fixed FPR grid for averaging across folds or sites. `load_raw` resolves a `(model, dim, cv, experiment)` combination to its artefact folder and loads it. `site_label` extracts the site name from an artefact filename for use in LOSO legends.

In [ ]:
VALID_TAGS      = {'SKF-Baseline', 'SKF-Optimised', 'LOSO-Baseline', 'LOSO-Optimised'}
ALL_MODELS      = ['SVM', 'RF', 'XGB', 'GCN']
ALL_DIMS        = ['SelectKBest', 'PCA', 'SDAE']
ALL_CV_SCHEMES  = ['SKF', 'LOSO']
ALL_EXPERIMENTS = ['Baseline', 'Optimised']

# Resolve FILTER
MODELS      = FILTER['models']      or ALL_MODELS
DIMS        = FILTER['dims']        or ALL_DIMS
CV_SCHEMES  = FILTER['cv']          or ALL_CV_SCHEMES
EXPERIMENTS = FILTER['experiments'] or ALL_EXPERIMENTS

print("Running with:")
print(f"  experiments : {EXPERIMENTS}")
print(f"  cv          : {CV_SCHEMES}")
print(f"  models      : {MODELS}")
print(f"  dims        : {DIMS}")
print()

## 6. SKF ROC Curves

One figure per feature strategy. All four models are overlaid on a single panel. The plotted curve is the mean across folds. Uncomment the individual fold lines inside `plot_skf_roc` to show fold-level curves.

In [ ]:
if 'SKF' in CV_SCHEMES:
    print('--- SKF ROC curves ---')
    for exp in EXPERIMENTS:
        for dim in DIMS:
            plot_skf_roc(dim=dim, exp=exp, models=MODELS, runs=RUNS, model_colours=MODEL_COLOURS, dim_labels=DIM_LABELS, save=True)

## 7. LOSO ROC Curves

One figure per feature strategy per model. Individual site curves are shown in distinct colours with site names and AUC values in the legend. Uncomment the mean curve block to overlay the mean across sites.

In [ ]:
if 'LOSO' in CV_SCHEMES:
    print('--- LOSO ROC curves ---')
    for exp in EXPERIMENTS:
        for dim in DIMS:
            for model in MODELS:
                plot_loso_roc(dim=dim, model=model, exp=exp, runs=RUNS, model_colours=MODEL_COLOURS, dim_labels=DIM_LABELS, save=True)